In [ ]:
from google.cloud import bigquery
from google.colab import auth

# 구글 계정 인증
auth.authenticate_user()

# BigQuery 연결
PROJECT_ID = "your-gcp-project"
client = bigquery.Client(project=PROJECT_ID)

print("BigQuery 연결 완료")

# PING 24시간 Feature Mart

공통 BASE 사용자의 최초 PING 이후 24시간 동안의
PING 발신·수신 및 관계망 행동을 사용자 단위로 집계한다.

## Feature

- `ping_sent_count_24h`: 첫 24시간 발신 횟수
- `ping_received_count_24h`: 첫 24시간 수신 횟수
- `unique_sent_users_24h`: 발신한 고유 상대 수
- `unique_received_users_24h`: 수신한 고유 상대 수
- `active_hours_24h`: 발신 활동이 발생한 고유 시간대 수
- `send_receive_ratio_24h`: 발신 1건당 수신 PING 수

모든 행동은 각 사용자의
`feature_start_at <= created_at < feature_end_at`
구간만 사용한다.

In [ ]:
ping_feature_query = """
WITH sent AS (
    SELECT
        b.user_id,

        COUNT(q.user_id) AS ping_sent_count_24h,

        COUNT(
            DISTINCT CAST(q.chosen_user_id AS INT64)
        ) AS unique_sent_users_24h,

        COUNT(
            DISTINCT DATETIME_TRUNC(
                DATETIME(q.created_at, 'Asia/Seoul'),
                HOUR
            )
        ) AS active_hours_24h

    FROM `your-gcp-project.mart.mart_ml_d7_base` b

    LEFT JOIN `your-gcp-project.raw.raw_userquestionrecord` q
        ON b.user_id = CAST(q.user_id AS INT64)
        AND q.created_at >= b.feature_start_at
        AND q.created_at < b.feature_end_at

    GROUP BY b.user_id
),

received AS (
    SELECT
        b.user_id,

        COUNT(q.user_id) AS ping_received_count_24h,

        COUNT(
            DISTINCT CAST(q.user_id AS INT64)
        ) AS unique_received_users_24h

    FROM `your-gcp-project.mart.mart_ml_d7_base` b

    LEFT JOIN `your-gcp-project.raw.raw_userquestionrecord` q
        ON b.user_id = CAST(q.chosen_user_id AS INT64)
        AND q.created_at >= b.feature_start_at
        AND q.created_at < b.feature_end_at

    GROUP BY b.user_id
)

SELECT
    b.user_id,

    COALESCE(s.ping_sent_count_24h, 0)
        AS ping_sent_count_24h,

    COALESCE(r.ping_received_count_24h, 0)
        AS ping_received_count_24h,

    COALESCE(s.unique_sent_users_24h, 0)
        AS unique_sent_users_24h,

    COALESCE(r.unique_received_users_24h, 0)
        AS unique_received_users_24h,

    COALESCE(s.active_hours_24h, 0)
        AS active_hours_24h,

    SAFE_DIVIDE(
        COALESCE(r.ping_received_count_24h, 0),
        NULLIF(s.ping_sent_count_24h, 0)
    ) AS send_receive_ratio_24h

FROM `your-gcp-project.mart.mart_ml_d7_base` b

LEFT JOIN sent s
    USING(user_id)

LEFT JOIN received r
    USING(user_id)

ORDER BY b.user_id
"""

ping_df = client.query(
    ping_feature_query
).to_dataframe()

print("행 수:", len(ping_df))
print("고유 사용자:", ping_df["user_id"].nunique())
print("중복:", ping_df["user_id"].duplicated().sum())

ping_df.head()

In [ ]:
ping_df = client.query(
    ping_feature_query
).to_dataframe()

print("행 수:", len(ping_df))
print("고유 사용자:", ping_df["user_id"].nunique())
print("중복:", ping_df["user_id"].duplicated().sum())

ping_df.head()

In [ ]:
ping_df[
    [
        "ping_sent_count_24h",
        "ping_received_count_24h",
        "unique_sent_users_24h",
        "unique_received_users_24h",
        "active_hours_24h",
        "send_receive_ratio_24h"
    ]
].describe().round(2)

In [ ]:
print(
    "발신 고유 상대 > 발신 횟수:",
    (ping_df["unique_sent_users_24h"] >
     ping_df["ping_sent_count_24h"]).sum()
)

print(
    "수신 고유 상대 > 수신 횟수:",
    (ping_df["unique_received_users_24h"] >
     ping_df["ping_received_count_24h"]).sum()
)

print(
    "발신 0:",
    (ping_df["ping_sent_count_24h"] == 0).sum()
)

print(
    "NULL:",
    ping_df.isnull().sum().sum()
)

In [ ]:
ping_df.nlargest(
    10,
    "send_receive_ratio_24h"
)[[
    "user_id",
    "ping_sent_count_24h",
    "ping_received_count_24h",
    "unique_sent_users_24h",
    "unique_received_users_24h",
    "send_receive_ratio_24h"
]]

# PING Mart 저장

In [ ]:
create_ping_mart_query = f"""
CREATE OR REPLACE TABLE
`your-gcp-project.mart.fct_ml_ping_24h`
AS

{ping_feature_query}
"""

client.query(create_ping_mart_query).result()

print("fct_ml_ping_24h 생성 완료")

In [ ]:
check_query = """
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT user_id) AS user_count,
    COUNT(*) - COUNT(DISTINCT user_id) AS duplicate_count
FROM `your-gcp-project.mart.fct_ml_ping_24h`
"""

client.query(check_query).to_dataframe()